In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from lib.data_quality import data_quality_report, apply_data_quality_rules
from lib.feature_selection import classification_woe_iv
from lib.probability_of_default import ProbabilityOfDefault
from lib.feature_selection import WoETransformer
from lib.feature_engineering import select_columns, log_loan_amount

# Load Data and check it's quality

In [ ]:
df = pd.read_csv("data/credit_risk_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe(include="all")

In [ ]:
data_quality_report(df)

The data has 165 duplicated rows, which will cause overoptimistic test results after splitting the data, and has some data quality issues (impossible/unlikely employment length, and high age).

The quality rules below set outliers to null, as they are MNAR and the model can account for this, and removes duplicated rows.

In [ ]:
df.drop_duplicates(inplace=True)
df = apply_data_quality_rules(df)
data_quality_report(df)

Split the data into train and test sets for evaluation. Each split set has it's index reset so they can interact with numpy arrays later more cleanly (particularly the calibration evaluation).

In [ ]:
target_col = "loan_status"

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

X_train = X_train.reset_index(drop=True).copy()
X_test = X_test.reset_index(drop=True).copy()
y_train = y_train.reset_index(drop=True).copy()
y_test = y_test.reset_index(drop=True).copy()

In [ ]:
print(
    "      |   X   | y\n"
    "––––––––––––––––––––––––\n"
    f"Train | {len(X_train)} | {len(y_train)}\n"
    f"Test  | {len(X_test)}  | {len(y_test)}\n"
)

In [ ]:
fig, ax = plt.subplots()

(y_train.value_counts(normalize=True)*100).plot(kind="bar", title="Default distribution in training set", ax=ax)

ax.set_xlabel("Loan Status (Default=1)")
ax.set_ylabel("Percent (%)")

plt.show()

# Feature selection and engineering

In [ ]:
iv, woe = classification_woe_iv(pd.concat([X_train, y_train], axis=1), target_col, show_woe=False)
woe

The below features are selected because they have an information value (IV) that suggests that they are good predictors of default. Note that loan_grade (IV=0.87) and loan_int_rate (IV=0.66) are not included as features in the modelling. This is because these features are circular and also are part of seperate modelling within the same process as PD.

In [ ]:
feats = ["person_income", "person_home_ownership", "person_emp_length", "loan_intent",
         "loan_amnt", "loan_percent_income", "cb_person_default_on_file"]

In [ ]:
woe_feats = [f for f in feats if f!="loan_amnt"]

In [ ]:
pipe = Pipeline([
    ("select",     FunctionTransformer(select_columns, kw_args={"columns": feats})),
    ("log_amount", FunctionTransformer(log_loan_amount)),
    ("woe",        WoETransformer(variables=woe_feats)),
    ("model",      LogisticRegression(class_weight="balanced", random_state=42)),
])

# Probability of Default Modelling and Evaluation

In [ ]:
pod = ProbabilityOfDefault(pipe, X_train, X_test, y_train, y_test)

## Cross-validation

In [ ]:
pod.cross_validation_metrics()

In [ ]:
pod.run_cv_calibration_tests()

## Model discrimination on test set

In [ ]:
print(f"ROC AUC: {pod.roc_auc():.2f}")

In [ ]:
print(f"PR AUC:   {pod.pr_auc():.2f}")

In [ ]:
print(f"Average precision:   {pod.average_precision():.2f}")

In [ ]:
ginis = []
for seed in range(10):
    a, b, c, d = train_test_split(X, y, stratify=y, test_size=0.2, random_state=seed)
    ginis.append(ProbabilityOfDefault(pipe, a, b, c, d).gini_coefficient())

print(f"Test Gini: {np.mean(ginis):.3f} ± {np.std(ginis):.3f}")

In [ ]:
print(f"Gini coefficient:   {pod.gini_coefficient():.2f}")

In [ ]:
print(f"KS Statistic: {pod.ks_statistic():.2f}")

In [ ]:
print(pod.generate_ranking_stability_report())

In [ ]:
print(f"Brier score:   {pod.brier_score():.2f}")

## Model calibration on test set

In [ ]:
hl_stat, dof, p_value = pod.hosmer_lemeshow_test()
hl_df = pd.DataFrame({
    "Hosmer-Lemeshow Statistic": [hl_stat],
    "Degrees of Freedom": [dof],
    "p-value": [p_value]
})
hl_df

In [ ]:
print(pod.binomial_goodness_of_fit_test(strategy='quantile'))

In [ ]:
fig, ax = plt.subplots()

ax.plot(np.linspace(0, 1, 100), np.linspace(0, 1, 100), color="black", linestyle="--")
pod.binomial_goodness_of_fit_test().plot(x="expected_prob", y="observed_prob", title="LR Calibration", ax=ax)

plt.show()

## Population Stability Index

In [ ]:
psi_report = pod.generate_psi_report(n_bins=10)
print(psi_report.to_string(index=False))

In [ ]:
pod.fit()
test_pds = pod.calibrated_model.predict_proba(X_test)[:, 1]
test_scores = pod.convert_pd_to_credit_score(test_pds)

# Quick verification display mapping PD to Scores
score_samples = pd.DataFrame({
    'Actual Label': y_test,
    'Calibrated PD (%)': np.round(test_pds * 100, 1),
    'Credit Bureau Point Score': test_scores
})
print("\n=== Credit Scoring Samples (Test Set) ===")
print(score_samples.head(10))